In [38]:
import pandas as pd
import numpy as np

import warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore', module='sklearn.preprocessing')

## Prepare data
- Display
    Display illustrate the data we are working with 
- X and y meaning
    - X: represents the columns that we need to
    - y: represents the prediction we want the model to target


In [39]:
apartments = pd.read_csv("challenge1_train.csv")
apartments_working = apartments.copy()

apartments_working['square_feet'] = np.log1p(apartments_working['square_feet'])
apartments_working['total_rooms'] = apartments_working['bedrooms'] + apartments_working['bathrooms']
apartments_working['sqft_per_room'] = apartments_working['square_feet'] / apartments_working['total_rooms']

coords = apartments_working[['latitude', 'longitude']]
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
apartments_working['neighborhood_cluster'] = kmeans.fit_predict(coords).astype(str)

display(apartments_working.head())
print(apartments_working.columns.tolist())
apartments_working.info()


,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state,total_rooms,sqft_per_room,neighborhood_cluster
0,2,925,1.0,0,4.762174,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA,1.0,4.762174,4
1,3,2475,1.0,0,4.875197,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY,1.0,4.875197,2
2,5,1695,1.0,0,5.252273,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA,1.0,5.252273,1
3,6,1560,1.0,1,5.303305,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC,2.0,2.651652,2
4,7,1560,1.0,1,5.303305,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC,2.0,2.651652,2


['ID', 'price', 'bathrooms', 'bedrooms', 'square_feet', 'latitude', 'longitude', 'category', 'has_photo', 'state', 'total_rooms', 'sqft_per_room', 'neighborhood_cluster']
<class 'pandas.DataFrame'>
RangeIndex: 7398 entries, 0 to 7397
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    7398 non-null   int64  
 1   price                 7398 non-null   int64  
 2   bathrooms             7398 non-null   float64
 3   bedrooms              7398 non-null   int64  
 4   square_feet           7398 non-null   float64
 5   latitude              7398 non-null   float64
 6   longitude             7398 non-null   float64
 7   category              7398 non-null   str    
 8   has_photo             7398 non-null   str    
 9   state                 7398 non-null   str    
 10  total_rooms           7398 non-null   float64
 11  sqft_per_room         7398 non-null   float64
 12  neighborhood_c

In [40]:

X = apartments_working.drop(columns=["price", "ID", "category"])
y = apartments_working["price"].copy()

numerical = ["square_feet", "bedrooms", "bathrooms", "latitude", "longitude", "total_rooms", "sqft_per_room"]
categorical = ["has_photo", "state", "neighborhood_cluster"] 

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical)
    ]
)

pipelines = {
    "Linear Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LinearRegression()),
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeRegressor(max_depth=14, 
                                        ccp_alpha=0.2, 
                                        max_features=None, 
                                        max_leaf_nodes=None,
                                        min_samples_split=18,
                                        min_samples_leaf=7,  
                                        splitter="best", 
                                        criterion="squared_error", 
                                        random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(n_estimators=350, 
                                        max_depth=30,
                                        min_samples_split=3,
                                        min_samples_leaf=1,
                                        max_features='sqrt',
                                        bootstrap=True,
                                        random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("model", XGBRegressor(random_state=42))
    ])
}

### Decision Tree 
    Established parameters by using randomized grid search
    Managed to lower RMSE from 550 to 481

In [ ]:
param_grid = {
    'model__max_depth': range(2, 20, 2),
    'model__min_samples_split': range(2, 20, 2),
    'model__min_samples_leaf': range(1, 20, 2),
    'model__max_features': [None, 'sqrt', 'log2'],
    'model__max_leaf_nodes': [None, 10, 20, 30, 40],
    'model__ccp_alpha': np.arange(0.0, 1.0, 0.1)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_DT = RandomizedSearchCV(
    estimator=pipelines["Decision Tree"],
    param_distributions=param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_iter=100, 
    n_jobs=-1
)

grid_search_DT.fit(X_train, y_train)


print(f"Best parameters: {grid_search_DT.best_params_}")
print(f"Best CV RMSE: {-grid_search_DT.best_score_:.3f}")


Best parameters: {'model__min_samples_split': 18, 'model__min_samples_leaf': 7, 'model__max_leaf_nodes': None, 'model__max_features': None, 'model__max_depth': 14, 'model__ccp_alpha': np.float64(0.2)}
Best CV RMSE: 481.826


In [41]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

pipelines["Decision Tree"].fit(X_train, y_train)
prediction = pipelines["Decision Tree"].predict(X_valid)
scores = cross_val_score(pipelines["Decision Tree"], X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')


print(f"Decision Tree - RMSE: {np.sqrt(mean_squared_error(y_valid, prediction)):.3f}")
print(f"Decision Tree - CV RMSE: {-scores.mean():.3f}")

Decision Tree - RMSE: 517.284
Decision Tree - CV RMSE: 481.871


### Random Forest

In [8]:
param_grid_RF = {
    'model__n_estimators': [250, 300, 350],
    'model__max_depth': [25, 30, 35],
    'model__min_samples_split': [3],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt'],
    'model__bootstrap': [True]
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_RF = RandomizedSearchCV(
    estimator=pipelines["Random Forest"],
    param_distributions=param_grid_RF,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_iter=100,
    n_jobs=-1
)

grid_search_RF.fit(X_train, y_train)

print(f"Best parameters: {grid_search_RF.best_params_}")
print(f"Best CV RMSE: {-grid_search_RF.best_score_:.3f}")


Best parameters: {'model__n_estimators': 350, 'model__min_samples_split': 3, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 30, 'model__bootstrap': True}
Best CV RMSE: 412.815


In [42]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

pipelines["Random Forest"].fit(X_train, y_train)
prediction = pipelines["Random Forest"].predict(X_valid)
scores = cross_val_score(pipelines["Random Forest"], X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')

print(f"Random Forest - RMSE: {np.sqrt(mean_squared_error(y_valid, prediction)):.3f}")
print(f"Random Forest - CV RMSE: {-scores.mean():.3f}")

Random Forest - RMSE: 455.802
Random Forest - CV RMSE: 427.639


### XGBoost

In [44]:
param_grid_XGB = {
    'model__n_estimators': [450, 500, 550],
    'model__max_depth': [4, 5, 6],
    'model__learning_rate': [0.08, 0.1, 0.12],
    'model__rate_drop': [0.1],
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_XGB = RandomizedSearchCV(
    estimator=pipelines["XGBoost"],
    param_distributions=param_grid_XGB,
    cv=kf,
    n_iter=10,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

grid_search_XGB.fit(X_train, y_train)

print(f"Best parameters: {grid_search_XGB.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search_XGB.best_score_):.3f}")

Best parameters: {'model__rate_drop': 0.1, 'model__n_estimators': 450, 'model__max_depth': 6, 'model__learning_rate': 0.12}
Best CV RMSE: 436.092


## Results

In [43]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_valid)
    print(f"{model_name} - RMSE: {np.sqrt(mean_squared_error(y_valid, prediction)):.3f}")

Linear Regression - RMSE: 556.054
Decision Tree - RMSE: 517.284
Random Forest - RMSE: 455.802
XGBoost - RMSE: 428.854


## Visualization
